# cap10_wf2020_participation_failure_attribution_v1

Purpose: diagnose why the current cap10 metric-replay handoff still cannot produce a retained walk-forward model, and convert the failed fold into a concrete next-training target contract.

Governance:
- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `cap10_three_fold_checkpoint_metric_replay_v2`.
- Role: `diagnose`, not promotion.
- Target window: `wf_2020` failure inside the required `wf_2020`, `wf_2022`, `wf_2023_2024` retention set.
- Accepted trade-off: identify a training target; do not retain an actor until all folds pass `relative_return >= 30pp` and `MaxDD < 20%`, plus YTD return alignment.
- Changed layer: `model_selection`, `failure_attribution`, `target_contract`.
- External reference: SOXLBacktest Qlib buy/sell reference and current-best retention thresholds.
- Hypothesis: `wf_2020` is blocked by bull/rebound underparticipation, not drawdown; checkpoint selection alone cannot close the relative-return gap.
- Failure mode: if the best checkpoint still leaves a large relative-return gap, next work must rebuild fold-local bull participation targets before PPO/retention.
- Comparison row key: `cap10_wf2020_participation_failure_attribution_v1`.

Historical context:
- Closest success case: `cap10_three_fold_checkpoint_metric_replay_v2` selected candidates pass `wf_2022` and `wf_2023_2024` under the replay-aligned fold gate.
- Closest failure case: the same card fails `wf_2020` with low relative return despite acceptable selected MaxDD.
- Non-duplicate reason: this notebook does not rerun checkpoint selection; it classifies the failed fold and writes the target contract that the next full-retention training branch must satisfy.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in ROOT.parents:
        if (parent / "src").exists():
            ROOT = parent
            break
for candidate in (ROOT, ROOT / "src"):
    text = str(candidate)
    if text not in sys.path:
        sys.path.insert(0, text)

from src.soxl_quant.app.walkforward_failure_attribution import (
    DEFAULT_TARGET_MAX_DD_PCT,
    DEFAULT_TARGET_RELATIVE_RETURN_PP,
    classify_walkforward_checkpoint_failures,
    summarize_walkforward_failure_attribution,
)

CARD = ROOT / "runtime/model_selection/card/cap10_wf2020_participation_failure_attribution_v1"
SOURCE = ROOT / "runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2"
CANDIDATES_PATH = SOURCE / "checkpoint_candidate_metrics.csv"
SELECTED_PATH = SOURCE / "checkpoint_selection_by_fold.csv"

CARD.mkdir(parents=True, exist_ok=True)
assert CANDIDATES_PATH.exists(), CANDIDATES_PATH
assert SELECTED_PATH.exists(), SELECTED_PATH

In [2]:
candidate_metrics = pd.read_csv(CANDIDATES_PATH)
selected_by_fold = pd.read_csv(SELECTED_PATH)

attribution = classify_walkforward_checkpoint_failures(
    candidate_metrics,
    selected_by_fold,
    target_relative_return_pp=DEFAULT_TARGET_RELATIVE_RETURN_PP,
    target_max_dd_pct=DEFAULT_TARGET_MAX_DD_PCT,
)
summary = summarize_walkforward_failure_attribution(attribution)

attribution

,fold,selected_candidate,selected_candidate_gate,failure_type,selected_relative_return_pp,selected_relative_gap_pp,selected_max_dd_pct,selected_drawdown_gap_pct,selected_test_return_pct,selected_buyhold_return_pct,selected_return_vs_buyhold_gap_pp,best_relative_candidate,best_relative_return_pp,best_relative_gap_pp,best_relative_max_dd_pct,best_relative_max_dd_gap_pct,best_relative_test_return_pct,candidate_gate_pass_count,checkpoint_selection_exhausted,next_training_target
0,wf_2020,bc_initialization,False,bull_underparticipation,-292.753021,322.753021,14.984701,0.0,40.580326,333.333347,-292.753021,ppo_retained_iter_4,-272.009298,302.009298,20.918285,0.918285,61.324050,0,True,bull_rebound_participation_lift_with_drawdown_...
1,wf_2022,bc_initialization,True,pass,65.864820,0.000000,18.802284,0.0,-12.708336,-78.573156,65.864820,ppo_retained_iter_4,71.293066,0.000000,10.123049,0.000000,-7.280090,4,False,none
2,wf_2023_2024,ppo_retained_iter_4,True,pass,35.566229,0.000000,8.955769,0.0,-0.625472,-36.191701,35.566229,ppo_retained_iter_4,35.566229,0.000000,8.955769,0.000000,-0.625472,2,False,none


In [3]:
failed = attribution[attribution["failure_type"].ne("pass")].copy()
contract_columns = [
    "fold",
    "failure_type",
    "selected_candidate",
    "selected_relative_return_pp",
    "selected_relative_gap_pp",
    "selected_max_dd_pct",
    "selected_drawdown_gap_pct",
    "selected_return_vs_buyhold_gap_pp",
    "best_relative_candidate",
    "best_relative_return_pp",
    "best_relative_gap_pp",
    "best_relative_max_dd_pct",
    "best_relative_max_dd_gap_pct",
    "checkpoint_selection_exhausted",
    "next_training_target",
]
if failed.empty:
    target_contract = pd.DataFrame(columns=contract_columns + ["acceptance_rule", "training_blocker"])
else:
    target_contract = failed[contract_columns].copy()
    target_contract["target_relative_return_pp"] = DEFAULT_TARGET_RELATIVE_RETURN_PP
    target_contract["target_max_dd_pct"] = DEFAULT_TARGET_MAX_DD_PCT
    target_contract["selected_max_dd_headroom_pct"] = (
        DEFAULT_TARGET_MAX_DD_PCT - target_contract["selected_max_dd_pct"]
    ).clip(lower=0.0)
    target_contract["observed_best_checkpoint_lift_pp"] = (
        target_contract["best_relative_return_pp"] - target_contract["selected_relative_return_pp"]
    )
    target_contract["remaining_gap_after_best_checkpoint_pp"] = target_contract["best_relative_gap_pp"]
    target_contract["acceptance_rule"] = "fold relative_return>=30pp and MaxDD<20pct before any retained actor"
    target_contract["training_blocker"] = np.where(
        target_contract["checkpoint_selection_exhausted"],
        "checkpoint_selection_cannot_close_gap",
        "candidate_selection_needs_review",
    )

target_contract

,fold,failure_type,selected_candidate,selected_relative_return_pp,selected_relative_gap_pp,selected_max_dd_pct,selected_drawdown_gap_pct,selected_return_vs_buyhold_gap_pp,best_relative_candidate,best_relative_return_pp,...,best_relative_max_dd_gap_pct,checkpoint_selection_exhausted,next_training_target,target_relative_return_pp,target_max_dd_pct,selected_max_dd_headroom_pct,observed_best_checkpoint_lift_pp,remaining_gap_after_best_checkpoint_pp,acceptance_rule,training_blocker
0,wf_2020,bull_underparticipation,bc_initialization,-292.753021,322.753021,14.984701,0.0,-292.753021,ppo_retained_iter_4,-272.009298,...,0.918285,True,bull_rebound_participation_lift_with_drawdown_...,30.0,20.0,5.015299,20.743723,302.009298,fold relative_return>=30pp and MaxDD<20pct bef...,checkpoint_selection_cannot_close_gap


In [4]:
def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value):
        return None
    return str(value)

created_at = datetime.now(timezone.utc).isoformat()
summary_payload = {
    **summary,
    "created_at": created_at,
    "experiment": "cap10_wf2020_participation_failure_attribution_v1",
    "decision": "wf2020_bull_participation_target_rebuild_required_no_promotion"
    if summary["primary_failure_type"] == "bull_underparticipation"
    else "walkforward_failure_attribution_no_promotion",
    "actor_retention_allowed": False,
    "promotion_allowed": False,
    "training_allowed_from_this_card": False,
    "role": "diagnose",
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "cap10_three_fold_checkpoint_metric_replay_v2",
    "reference_baseline": "soxlbacktest_qlib_buysell_reference",
    "changed_layer": ["model_selection", "failure_attribution", "target_contract"],
    "target_relative_return_pp": DEFAULT_TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": DEFAULT_TARGET_MAX_DD_PCT,
    "source_candidate_metrics": str(CANDIDATES_PATH.relative_to(ROOT)),
    "source_selection": str(SELECTED_PATH.relative_to(ROOT)),
    "sidecars": [
        "failure_attribution.csv",
        "bull_participation_target_contract.csv",
        "selection.json",
        "summary.json",
    ],
}
selection_payload = {
    **summary_payload,
    "failed_fold_rows": failed.to_dict(orient="records"),
    "target_contract_rows": target_contract.to_dict(orient="records"),
}

attribution.to_csv(CARD / "failure_attribution.csv", index=False)
target_contract.to_csv(CARD / "bull_participation_target_contract.csv", index=False)
(CARD / "summary.json").write_text(json.dumps(summary_payload, indent=2, sort_keys=True, default=json_default) + "\n", encoding="utf-8")
(CARD / "selection.json").write_text(json.dumps(selection_payload, indent=2, sort_keys=True, default=json_default) + "\n", encoding="utf-8")

summary_payload

{'created_at': '2026-06-13T21:43:33.752789+00:00',
 'promotion_allowed': False,
 'failed_folds': ['wf_2020'],
 'failed_fold_count': 1,
 'primary_failure_type': 'bull_underparticipation',
 'checkpoint_selection_exhausted_failed_count': 1,
 'max_selected_relative_gap_pp': 322.753021,
 'max_selected_drawdown_gap_pct': 0.0,
 'next_mechanism': 'rebuild_wf2020_bull_participation_target_before_full_retention_training',
 'experiment': 'cap10_wf2020_participation_failure_attribution_v1',
 'decision': 'wf2020_bull_participation_target_rebuild_required_no_promotion',
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'role': 'diagnose',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'cap10_three_fold_checkpoint_metric_replay_v2',
 'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
 'changed_layer': ['model_selection',
  'failure_attribution',
  'target_contract'],
 'target_relative_return_pp': 30.0,
 'target_max

## Conclusion

This diagnostic does not promote or retain any actor. The current cap10 metric-replay handoff has complete fold coverage, but `wf_2020` fails because relative return is far below the +30pp target while the selected candidate's MaxDD is already below 20%. The best observed checkpoint only improves the relative gap modestly and breaches the drawdown ceiling, so checkpoint selection is exhausted for this fold.

Next action: rebuild a fold-local `wf_2020` bull/rebound participation target with an explicit drawdown budget, then use that target as the source for the next BC/PPO or checkpoint scoring branch. Do not start actor retention until the next branch proves all three walk-forward windows and then passes YTD alignment against the current-best and SOXLBacktest Qlib reference baselines.